# 06. Sequence 분류 코드 흐름과 실행 결과

대상: `main.py` — RNN·LSTM·Transformer의 validation 예측, 오류 수, 파라미터 수, 학습시간 기록.

이 노트북은 현재 `outputs/predictions.csv`와 `outputs/comparison.csv`를 읽어 실행 결과를 확인합니다. 모델의 순위나 우수성에 대한 결론은 작성하지 않습니다.

필요 패키지: `torch`, `pandas`, `IPython`. 커널은 프로젝트의 `.venv` 등 패키지가 설치된 Python 환경을 선택하세요.

## 1. 전체 흐름

1. PyTorch CPU 스레드 수를 1로 고정하고 난수 시드를 설정합니다.
2. 제공 모듈에서 독립 Sequence의 train·valid 분할을 읽습니다.
3. RNN → LSTM → Transformer 순서로 모델을 만들고 각각 25 epoch 학습합니다.
4. 반환된 validation logits에서 `argmax(dim=1)`로 예측 라벨을 구합니다.
5. 실제 라벨과 비교해 표본별 `error`를 기록합니다.
6. 예측 상세와 모델별 측정값을 CSV에 저장하고 오류 수를 출력합니다.

원본 코드와 기존 CSV는 수정하지 않습니다. 아래 경로 탐색은 노트북 폴더 또는 프로젝트 내부에서 실행하는 경우를 지원합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch
from IPython.display import display

cwd = Path.cwd().resolve()
exercise = None
for base in [cwd, *cwd.parents]:
    for candidate in [base, base / "06_errors", base / "20261103/03_심화/06_errors"]:
        if (candidate / "main.py").exists() and (candidate.parent / "day12_sequence_models").is_dir():
            exercise = candidate
            break
    if exercise is not None:
        break
if exercise is None:
    raise FileNotFoundError("프로젝트 내부에서 노트북을 실행하세요.")

root = exercise.parent
sys.path.insert(0, str(root / "day12_sequence_models"))
from data import classification_data
from models import classifier
from train import fit

torch.set_num_threads(1)
print("대상:", exercise.name + "/main.py")

대상: 06_errors/main.py


## 2. 데이터 준비

`classification_data()`는 `data/sequences.json`에서 train·valid만 읽습니다. 길이가 다른 Sequence는 `pad_sequence(..., batch_first=True)`로 채웁니다. 각 분할에는 다음 항목이 있습니다.

| 항목 | 의미 |
|---|---|
| `x` | `[B, T, F]` 입력: 표본 수, 최대 길이, 특성 수 |
| `lengths` | 각 Sequence의 실제 길이 |
| `y` | 분류 정답 라벨 |
| `ids` | 예측 결과를 원래 표본과 연결할 ID |

test는 이 로더에서 반환하지 않으며, 현재 코드의 학습·평가에 사용하지 않습니다.

In [ ]:
data = classification_data(root / "data")
data_summary = pd.DataFrame([
    {"split": split, "samples": len(batch["ids"]),
     "input_shape": str(tuple(batch["x"].shape)),
     "min_length": int(batch["lengths"].min()),
     "max_length": int(batch["lengths"].max()),
     "class_0": int((batch["y"] == 0).sum()),
     "class_1": int((batch["y"] == 1).sum())}
    for split, batch in data.items()
])
display(data_summary)

,split,samples,input_shape,min_length,max_length,class_0,class_1
0,train,60,"(60, 12, 3)",6,12,30,30
1,valid,20,"(20, 12, 3)",6,12,10,10


## 3. 모델 생성과 학습

원본은 다음 순서로 반복합니다.

```python
for name in ["RNN", "LSTM", "Transformer"]:
    torch.manual_seed(42)
    result = fit(classifier(name), data, "classification", epochs=25, lr=0.01)
```

각 모델을 생성하기 직전에 시드를 다시 설정합니다. 같은 시드라도 모델 구조와 파라미터 수가 같아지는 것은 아닙니다.

| 모델 | 입력 처리와 분류 출력 |
|---|---|
| RNN | hidden=24, Packed Sequence의 실제 마지막 hidden → Linear(24, 2) |
| LSTM | hidden=12, Packed Sequence의 실제 마지막 hidden → Linear(12, 2) |
| Transformer | 3→8 투영 + 위치 임베딩 → Encoder 1층(head=2, FFN=16) → 유효 시점 평균 → Linear(8, 2) |

RNN·LSTM은 `pack_padded_sequence`로 Padding을 제외합니다. Transformer는 Padding key mask와 평균 집계 시 Padding query 제외를 함께 적용합니다. 위치 임베딩의 최대 길이는 64이며 파라미터 수에 포함됩니다.

`fit()`의 동작:
- Adam, 학습률 0.01, CrossEntropyLoss로 full-batch 학습을 25회 수행합니다.
- 매 epoch 업데이트 후 eval 모드에서 train과 valid를 평가합니다.
- 각 모델 내부에서 valid loss가 최소였던 epoch의 상태를 보관하고, 학습 후 복원해 validation 예측을 반환합니다.
- `seconds`는 full-batch 학습과 epoch별 평가, 상태 보관 등을 포함한 학습 루프 구간입니다. optimizer 생성, 데이터 로딩, 루프 뒤 상태 복원·최종 평가, CSV 저장은 해당 시간에서 제외됩니다.

이 상태 복원은 각 모델 내부의 epoch 선택 절차입니다. 여기서는 모델 간 선택이나 추천을 하지 않습니다.

## 4. D06: logits → 예측 라벨 → 오류 여부

```python
pred = result["prediction"].argmax(dim=1)
actual = data["valid"]["y"]
```

`result["prediction"]`은 `[validation 표본 수, 2]` 형태의 클래스 logits입니다. `argmax(dim=1)`은 각 표본에서 값이 가장 큰 클래스의 인덱스를 선택하므로 결과는 `[validation 표본 수]`입니다. 클래스 선택에 softmax를 먼저 적용할 필요는 없습니다.

원본은 `zip(ids, actual, pred)`로 같은 표본의 ID·정답·예측을 묶습니다. `error = bool(y != p)`는 오답일 때 `True`입니다. `predictions.csv`에는 정답과 오답을 모두 저장하며, 오류 행만 저장하는 것은 아닙니다.

다음은 라벨 변환 원리를 보여주는 **설명용 logits**입니다. 실제 실행 예측은 다음 절의 CSV를 사용합니다.

In [ ]:
example_logits = torch.tensor([[1.2, -0.3], [-0.2, 0.8]])
example_actual = torch.tensor([0, 0])
example_pred = example_logits.argmax(dim=1)
display(pd.DataFrame({
    "class_0_logit": example_logits[:, 0].tolist(),
    "class_1_logit": example_logits[:, 1].tolist(),
    "actual": example_actual.tolist(),
    "prediction": example_pred.tolist(),
    "error": (example_actual != example_pred).tolist(),
}))

,class_0_logit,class_1_logit,actual,prediction,error
0,1.2,-0.3,0,0,False
1,-0.2,0.8,0,1,True


## 5. 저장된 실행 결과

아래 표는 현재 생성된 CSV를 직접 읽은 결과입니다. `expected/`의 예시 값을 복사하지 않습니다.

- `comparison.csv`: 모델별 파라미터 수, 측정 시간(초), validation 정확도
- `predictions.csv`: 모델·표본 ID·정답·예측·오류 여부

학습시간은 실행 환경과 초기 실행 비용 등의 영향을 받는 단일 실행 관측값입니다. 이 기록만으로 반복 측정한 속도 차이를 확정하지 않습니다. 표는 원본 실행 순서로 표시합니다.

In [ ]:
comparison = pd.read_csv(exercise / "outputs/comparison.csv")
predictions = pd.read_csv(exercise / "outputs/predictions.csv")
if predictions["error"].dtype != bool:
    predictions["error"] = predictions["error"].astype(str).str.lower().map({"true": True, "false": False})
    if predictions["error"].isna().any():
        raise ValueError("error 열에 True/False 외의 값이 있습니다.")

display(comparison)
error_summary = predictions.groupby("model", sort=False).agg(
    valid_samples=("id", "size"), error_count=("error", "sum")
).reset_index()
error_summary["accuracy_from_rows"] = 1 - error_summary["error_count"] / error_summary["valid_samples"]
display(error_summary)
print("원본 코드와 같은 오류 수 출력:")
print(predictions.groupby("model")["error"].sum().to_string())

,model,params,seconds,valid_accuracy
0,RNN,746,2.816990,1.0
1,LSTM,842,0.054821,1.0
2,Transformer,1162,0.064651,1.0


,model,valid_samples,error_count,accuracy_from_rows
0,RNN,20,0,1.0
1,LSTM,20,0,1.0
2,Transformer,20,0,1.0


원본 코드와 같은 오류 수 출력:
model
LSTM           0
RNN            0
Transformer    0


## 6. 표본별 결과와 실제 오류 행

전체 예측을 확인하고 `error=True`인 행만 별도로 표시합니다. 오류가 없으면 0건으로 남깁니다. 설명용 logits 예시의 오류는 이 실제 결과에 포함하지 않습니다.

In [ ]:
display(predictions)
errors = predictions.loc[predictions["error"]].copy()
print("실제 오류 행 수:", len(errors))
display(errors)

,model,id,actual,prediction,error
0,RNN,valid_000,0,0,False
1,RNN,valid_001,1,1,False
2,RNN,valid_002,0,0,False
3,RNN,valid_003,1,1,False
4,RNN,valid_004,0,0,False
5,RNN,valid_005,1,1,False
6,RNN,valid_006,0,0,False
7,RNN,valid_007,1,1,False
8,RNN,valid_008,0,0,False
9,RNN,valid_009,1,1,False


,model,id,actual,prediction,error


실제 오류 행 수: 0


## 7. CSV 기록의 일관성 확인

표본 ID·정답이 실제 validation 데이터와 일치하는지, 오류 표시가 정답 비교 결과와 같은지, CSV의 정확도와 파라미터 수가 일치하는지 확인합니다. 이 확인은 CSV의 정합성 검증이며 모델 순위를 정하는 과정은 아닙니다.

In [ ]:
names = ["RNN", "LSTM", "Transformer"]
assert set(comparison["model"]) == set(names)
assert len(comparison) == len(names)
assert set(predictions["model"]) == set(names)
assert not predictions.duplicated(["model", "id"]).any()
assert (predictions["error"] == (predictions["actual"] != predictions["prediction"])).all()
labels_by_id = dict(zip(data["valid"]["ids"], data["valid"]["y"].tolist()))
for name in names:
    part = predictions.loc[predictions["model"] == name]
    assert len(part) == len(labels_by_id)
    assert set(part["id"]) == set(labels_by_id)
    assert (part["actual"] == part["id"].map(labels_by_id)).all()
    assert part["prediction"].isin([0, 1]).all()
    row = comparison.loc[comparison["model"] == name].iloc[0]
    assert abs((part["actual"] == part["prediction"]).mean() - row["valid_accuracy"]) < 1e-6
    # 검증용 모델 초기화가 기존 난수 상태에 영향을 주지 않도록 합니다.
    with torch.random.fork_rng():
        params = sum(p.numel() for p in classifier(name).parameters())
    assert params == row["params"]
    assert row["seconds"] >= 0
print("CSV 기록의 일관성 검증을 통과했습니다.")

CSV 기록의 일관성 검증을 통과했습니다.


## 8. 선택 사항: 동일 조건으로 재학습

기본값인 `RUN_TRAINING=False`에서는 저장된 결과만 확인합니다. 재학습하려면 `True`로 변경하세요. 원본 코드와 동일하게 25 epoch, 학습률 0.01로 학습하며 결과는 메모리에 보관합니다. 기존 CSV는 덮어쓰지 않습니다.

학습시간은 이번 재실행에서 측정한 값입니다. 저장된 결과와 구분할 수 있도록 별도의 표로 표시합니다.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    rerun_rows = []
    rerun_tables = []
    for name in ["RNN", "LSTM", "Transformer"]:
        torch.manual_seed(42)
        result = fit(classifier(name), data, "classification", epochs=25, lr=0.01)
        pred = result["prediction"].argmax(dim=1)
        actual = data["valid"]["y"]
        rerun_tables.extend(
            {"model": name, "id": i, "actual": int(y), "prediction": int(p), "error": bool(y != p)}
            for i, y, p in zip(data["valid"]["ids"], actual, pred)
        )
        rerun_rows.append({"model": name, "params": result["params"],
                           "seconds": result["seconds"], "valid_accuracy": result["valid_accuracy"]})
    rerun_comparison = pd.DataFrame(rerun_rows)
    rerun_predictions = pd.DataFrame(rerun_tables)
    display(rerun_comparison)
    print(rerun_predictions.groupby("model")["error"].sum().to_string())
else:
    print("재학습은 실행하지 않았습니다. 위 표는 기존 CSV의 기록입니다.")

재학습은 실행하지 않았습니다. 위 표는 기존 CSV의 기록입니다.


## 결과를 읽을 때 구분할 사항

- 현재 정확도는 validation 분할에서 계산한 값입니다. test 성능이나 다른 데이터에서의 결과를 의미하지 않습니다.
- 같은 분류 문제의 정확도·오류 수와 파라미터 수·학습시간은 서로 다른 정보를 제공합니다.
- 재구성 MSE는 별도 목표의 지표이므로 이 분류 정확도와 평균내지 않습니다.
- 저장된 실행 결과는 관측값으로 제시하며 모델 선택에 대한 결론은 포함하지 않습니다.

## 9. 개인적 견해

**요약**: LSTM이 학습량이 적으면서 학습 시간도 적어 효율적이라 판단했다. 실제로 학습량이 가장 낮은건 RNN이었으나 학습 시간이 약 2초대로 가장 길었으며, Transformer는 학습 시간이 LSTM과 비슷했으나 학습량이 약 1162로 가장 많아 학습량에 대한 효율성이 낮다고 판단했다.

RNN, LSTM, Transformer를 각각 비교했을 때 아래와 같은 결과가 나타났다.


- 학습량
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>model</th>
      <th>params</th>
      <th>seconds</th>
      <th>valid_accuracy</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>RNN</td>
      <td>746</td>
      <td>2.816990</td>
      <td>1.0</td>
    </tr>
    <tr>
      <th>1</th>
      <td>LSTM</td>
      <td>842</td>
      <td>0.054821</td>
      <td>1.0</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Transformer</td>
      <td>1162</td>
      <td>0.064651</td>
      <td>1.0</td>
    </tr>
  </tbody>
</table>

- 오류 수
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>model</th>
      <th>valid_samples</th>
      <th>error_count</th>
      <th>accuracy_from_rows</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>RNN</td>
      <td>20</td>
      <td>0</td>
      <td>1.0</td>
    </tr>
    <tr>
      <th>1</th>
      <td>LSTM</td>
      <td>20</td>
      <td>0</td>
      <td>1.0</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Transformer</td>
      <td>20</td>
      <td>0</td>
      <td>1.0</td>
    </tr>
  </tbody>
</table>

해당 보고서를 참고하기 앞서, 이 학습량 비교과정에서 오류 수는 판단 근거에 포함하지 않기로 했다.    
이유는 실습용으로 학습량이 적어 과적합에 대한 여부를 판단하기 어려워 생략하기로 했다. 물론, 실제 프로젝트를 진행할 경우 해당 부분은 충분한 학습량을 확보한 후 판단하는 것이 바람직하다.    
특히, 실제 프로젝트에서는 오류 수가 완전히 0으로 수렴했을 경우 과적합을 의심하는 것이 합리적이다.    

오류 수에 대한 예외 사항을 제외하고 개인적인 판단 근거는 아래와 같이 지정했다.
- [ ] 학습량의 효율성: 학습량 대비 학습시간이 가장 적은 모델을 선택.
- [ ] 학습량의 절대값: 학습량이 적은 모델을 선택.
- [ ] 학습시간: 학습시간이 적은 모델을 선택.

위와 같은 판단근거를 선택한 이유는 제한된 학습 시간과 학습량을 고려했을 때, 최대한 적은 시간과 적은 파라미터로 효율적인 모델을 선택하는 것이 바람직하다고 판단했기 때문이다.    

그래서 위의 판단 근거를 기준으로 했을 때, LSTM이 가장 효율적인 모델이라고 판단했다.    

첫번째로, 학습량의 효율성에 대해서는 RNN은 학습량이 746으로 세 모델 중 가장 적었으나 학습량이 2.8초로 가장 길어 학습량 대비 학습시간에 대한 효율이 떨어졌다고 판단했다. Transformer는 학습시간이 0.6초로 LSTM과 비슷했으나, 학습량이 1162로 가장 많아 학습시간 대비 학습량에 대한 효율이 떨어졌다고 판단했으므로, 학습량 대비 학습시간에 대한 효율성은 LSTM이 가장 적절하다고 판단했다.    

두번째로, 학습량의 절대값은 RNN이 746으로 가장 적었으나, 학습량 대비 학습시간에 대한 효율성이 떨어졌기 때문에, 학습량의 절대값은 LSTM이 가장 적절하다고 판단했다.    

마지막으로, 학습시간은 LSTM이 0.05초로 가장 적었기 때문에, 학습시간에 대한 효율성은 LSTM이 가장 적절하다고 판단했다.    

종합적으로 위 내용을 고려했을 때, LSTM이 가장 효율적인 모델이라고 판단했다.